# R10 · Block structure (supplementary Fig S1)

How small the covariance entries are that the block-diagonal layout leaves out: per dataset, the |correlation| of
every within-block and every between-block gene pair of every index tile (shrunk covariances, each tile in the gene
order and blocks of its niche, seed-0 layout), as histograms; and one breast tile's correlation matrix as the example.
Per entry only (3 Nov decision): no share of the squared correlation, no d_B vs whole-matrix ranking panel.

* **Prerequisites:** `R0_index_cache` (staged covariances + seed-0 layouts).
* **Writes:** `results/block_structure/tile_means.csv` (per tile, Fig S1B), `hist.csv` (|r| histograms, bins of 0.001), `summary.csv` (quantiles and
  fractions above 0.1 / 0.3 per dataset and part), `example.npz` (the breast example).
* **Runtime / memory:** a few minutes; < 8 GB (covariances memory-mapped).
  `sbatch experiments/common/run_nb.sbatch experiments/compute/R10_block_structure.ipynb`

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets.yaml").exists())
sys.path.insert(0, str(ROOT / "experiments" / "common"))
import numpy as np
import pandas as pd

import block_structure as bs
import data
import results_io as rio

TOPIC = "block_structure"
FORCE = False
OUT = rio.topic(TOPIC)
print(data.stamp())

{'tag': 'method-v2', 'describe': 'method-v2-3-g88fc62a-dirty', 'commit': '88fc62a'}


## Histograms per dataset

In [2]:
if FORCE or not (OUT / "hist.csv").exists():
    hists, summs = [], []
    for key in data.indexed_keys():
        h, s = bs.dataset(key)
        hists.append(h); summs.append(s)
        print(key, s[["part", "q50", "q90", "frac_above_0.1"]].to_dict("records"), flush=True)
    st = data.stamp()
    hist = pd.concat(hists, ignore_index=True).assign(tag=st["tag"])
    summ = pd.concat(summs, ignore_index=True).assign(tag=st["tag"])
    rio.write_csv(hist, TOPIC, "hist.csv", "|r| histogram (bin_lo, bin_hi, count) of the within-block and between-block "
                  "gene pairs of every index tile, per dataset; seed-0 layout, shrunk covariances in niche order")
    rio.write_csv(summ, TOPIC, "summary.csv", "per dataset and part (within / between): pairs, |r| quantiles q50 / q90 / "
                  "q99 (upper bin edge, bins of 0.001) and the fraction of pairs with |r| > 0.1 and > 0.3")
summ = pd.read_csv(OUT / "summary.csv")
summ.pivot(index="key", columns="part", values=["q50", "q90", "q99", "frac_above_0.1", "pairs_per_tile"]).round(3)

q50            q90            q99         \
part               between within between within between within   
key                                                               
brain_cancer         0.025  0.107   0.131  0.422   0.333  0.688   
breast_cancer        0.031  0.107   0.144  0.477   0.358  0.752   
kidney_nondiseased   0.011  0.030   0.068  0.324   0.215  0.742   
lung_cancer          0.024  0.066   0.118  0.356   0.310  0.652   
lymph_node           0.018  0.038   0.089  0.261   0.262  0.594   
lymph_node_5k        0.066  0.084   0.183  0.275   0.343  0.589   
pancreatic_cancer    0.028  0.071   0.133  0.365   0.331  0.607   
skin_melanoma        0.026  0.074   0.127  0.407   0.328  0.737   

                   frac_above_0.1        pairs_per_tile            
part                      between within        between    within  
key                                                                
brain_cancer                0.148  0.515      42387.594  3012.342  
breast_cancer               0.173  0.512      33185.348  3033.289  
kidney_nondiseased          0.053  0.295      42118.837  2474.980  
lung_cancer                 0.128  0.410      40267.087  3246.424  
lymph_node                  0.083  0.279      45766.088  3344.120  
lymph_node_5k               0.310  0.429      73481.344  4521.477  
pancreatic_cancer           0.154  0.430      56131.999  2984.432  
skin_melanoma               0.144  0.433      45805.883  3170.548

## Breast example tile

In [3]:
if FORCE or not (OUT / "example.npz").exists():
    ex = bs.example("breast_cancer")
    np.savez_compressed(OUT / "example.npz", **ex)
    rio.register(TOPIC, OUT / "example.npz", "breast example: correlation matrix (float32) of the index tile whose "
                 "median between-block |r| is the median of the largest niche, in niche order; block runs, niche, "
                 "tile, genes")
z = np.load(OUT / "example.npz")
print(f"niche {int(z['niche'])}, tile {int(z['tile'])}, {z['corr'].shape[0]} genes, {len(z['runs'])} blocks")

niche 0, tile 850, 313 genes, 16 blocks


## Niche means (Fig S1, redrawn 3 Nov)
|r| over each niche's mean correlation matrix (from `niche_means.npz`), full matrix and inside blocks, pooled over the
niches; the breast example is the largest niche's matrix. Writes `niche_hist.csv` and `niche_example.npz`.

In [4]:
if FORCE or not (OUT / "niche_hist.csv").exists():
    hs = []
    for key in data.indexed_keys():
        h, ex = bs.niche_means(key)
        hs.append(h)
        if key == "breast_cancer":
            np.savez_compressed(OUT / "niche_example.npz", **ex)
            rio.register(TOPIC, OUT / "niche_example.npz", "breast: the largest niche's mean correlation matrix "
                         "(float32) in its gene order, block runs, niche, tiles")
    rio.write_csv(pd.concat(hs, ignore_index=True).assign(tag=data.stamp()["tag"]), TOPIC, "niche_hist.csv",
                  "|r| histogram (bins of 0.001) over each niche's mean correlation matrix, pooled over the niches: "
                  "full matrix (every off-diagonal pair) and blocks (pairs inside a block); seed-0 layout")
nh = pd.read_csv(OUT / "niche_hist.csv")
near = nh[nh.bin_hi <= 0.05 + 1e-9].groupby(["key", "part"])["count"].sum() / nh.groupby(["key", "part"])["count"].sum()
(100 * near).unstack().round(1)

part,blocks,full
key,,
brain_cancer,57.7,91.9
breast_cancer,39.3,80.7
kidney_nondiseased,58.0,94.7
lung_cancer,49.6,91.7
lymph_node,63.0,95.3
lymph_node_5k,32.1,63.9
pancreatic_cancer,59.3,91.1
skin_melanoma,48.6,89.5


## Per-tile means (Fig S1B, 3 Nov)
Each index tile's mean |r| over its within-block and its between-block gene pairs (pairs with a gene without counts
in the tile left out), every dataset. Writes `tile_means.csv`.

In [5]:
if FORCE or not (OUT / "tile_means.csv").exists():
    tm = pd.concat([bs.dataset_means(key) for key in data.indexed_keys()], ignore_index=True)
    rio.write_csv(tm.assign(tag=data.stamp()["tag"]), TOPIC, "tile_means.csv",
                  "one row per index tile (seed-0 layout): niche, mean |r| over its within-block and between-block gene "
                  "pairs (pairs with a gene without counts in the tile left out), pair counts")
tm = pd.read_csv(OUT / "tile_means.csv")
g = tm.groupby("key")
pd.DataFrame({"tiles": g.size(), "within": g.mean_within.mean(), "within_sd": g.mean_within.std(),
              "between": g.mean_between.mean(), "between_sd": g.mean_between.std(),
              "within > between (%)": 100 * g.apply(lambda d: (d.mean_within > d.mean_between).mean())}).round(3)

,tiles,within,within_sd,between,between_sd,within > between (%)
key,,,,,,
brain_cancer,9558,0.165,0.027,0.051,0.008,100.0
breast_cancer,1840,0.184,0.042,0.058,0.014,100.0
kidney_nondiseased,705,0.104,0.019,0.027,0.012,100.0
lung_cancer,1535,0.131,0.025,0.048,0.012,100.0
lymph_node,2354,0.093,0.023,0.038,0.013,100.0
lymph_node_5k,6972,0.122,0.025,0.085,0.011,100.0
pancreatic_cancer,2061,0.135,0.023,0.054,0.014,100.0
skin_melanoma,839,0.148,0.030,0.052,0.018,100.0


## Consensus tree and dyadic example (Figs S6, S7; 4 Nov)
Writes `consensus_example.npz` (S6) and `dyadic_example.npz` (S7) from the seed-0 layout, staged covariances and Spindle-Partial index.

In [6]:
if FORCE or not (OUT / "consensus_example.npz").exists():
    np.savez_compressed(OUT / "consensus_example.npz", **bs.consensus_example("breast_cancer"))
    rio.register(TOPIC, OUT / "consensus_example.npz", "Fig S6, breast largest niche (seed 0): three random tiles' gene "
                 "dendrograms (linkage, leaves), the consensus linkage (mean cophenetic matrix; leaf order == layout), "
                 "blocks, clade flag and two marker genes per block, mean correlation and gene names in niche order")
if FORCE or not (OUT / "dyadic_example.npz").exists():
    np.savez_compressed(OUT / "dyadic_example.npz", **bs.dyadic_example("breast_cancer"))
    rio.register(TOPIC, OUT / "dyadic_example.npz", "Fig S7, breast largest niche, its most co-varying block of 20-32 genes (seed 0 Spindle-Partial): "
                 "every aligned dyadic interval (s, j, K_I), an illustrative gene set's pieces and node counts, every "
                 "tile's node in every piece, the block's mean correlation and gene names")
kz, dz = np.load(OUT / "consensus_example.npz"), np.load(OUT / "dyadic_example.npz")
print(f"consensus: niche {int(kz['niche'])}, {int(kz['n_tiles'])} tiles, {len(kz['blocks'])} blocks, "
      f"{int(kz['is_clade'].sum())} of them subtrees: {kz['markers'][kz['is_clade']].tolist()}")
print("pieces (s, j):", dz["pieces"].tolist(), "K:", dz["piece_K"].tolist())

consensus: niche 0, 664 tiles, 16 blocks, 4 of them subtrees: ['CD3E PTPRC', 'FCER1G CD68', 'LUM POSTN', 'KRT7 ERBB2']
pieces (s, j): [[2, 2], [4, 4], [1, 8], [1, 11], [1, 12], [4, 16]] K: [128, 128, 32, 32, 32, 128]


In [7]:
rio.write_manifest(TOPIC, "R10_block_structure.ipynb", inputs=["results/_cache/<key>/covs.npy (R0)",
                                                               "results/_cache/<key>/seed0/layout.pkl (R0)",
                                                               "results/_cache/<key>/seed0/niche_means.npz (R0)",
                                                               "results/_cache/breast_cancer/seed0/spindle_partial.pkl (R0)"])

wrote results/block_structure/manifest.json: 9 files; peak memory 0.5 GB


{'topic': 'block_structure',
 'notebook': 'R10_block_structure.ipynb',
 'date': '2026-11-04 05:19',
 'tag': 'method-v2',
 'describe': 'method-v2-3-g88fc62a-dirty',
 'commit': '88fc62a',
 'inputs': ['results/_cache/<key>/covs.npy (R0)',
  'results/_cache/<key>/seed0/layout.pkl (R0)',
  'results/_cache/<key>/seed0/niche_means.npz (R0)',
  'results/_cache/breast_cancer/seed0/spindle_partial.pkl (R0)'],
 'peak_rss_gb': 0.5,
 'notes': '',
 'files': {'consensus_example.npz': {'description': "Fig S6, breast largest niche (seed 0): three random tiles' gene dendrograms (linkage, leaves), the consensus linkage (mean cophenetic matrix; leaf order == layout), blocks, clade flag and two marker genes per block, mean correlation and gene names in niche order",
   'rows': None,
   'bytes': 386304},
  'dyadic_example.npz': {'description': "Fig S7, breast largest niche, its most co-varying block of 20-32 genes (seed 0 Spindle-Partial): every aligned dyadic interval (s, j, K_I), an illustrative gene set'